# NSCG + O*NET Data Integration

## Summary

The integrated subset contains **6,849 respondents across 25 occupations**, with one row per person and **42 columns**. It includes nine NSCG importance items, six O*NET Work Values, original satisfaction codes and survey weights.

Two **left joins** keep NSCG respondents as the left table: `N3OCPR` links to the reviewed occupation concordance, then the screened `ONET_SOC_CODE` links to O*NET Work Values. Both use `many_to_one` validation and preserve all **79,800 eligible rows**. Only matches passing the supervision screen enter the saved subset (**8.6%**).

O*NET ratings describe occupations, not individual workplaces. Personal importance and occupational ratings remain separate measures; no fit score or model is included.

CSV: [nscg2023_onet30_2_subset.csv](nscg2023_onet30_2_subset.csv).

### Data preparation

Use Python 3.12 with these tested packages in the notebook's environment:

```bash
python -m pip install numpy==2.2.1 pandas==2.2.3 openpyxl==3.1.5 ipython==9.9.0 ipykernel==7.1.0
```

1. Download the [NSCG 2023 public-use ZIP](https://ncses.nsf.gov/822/assets/0/files/college_grads_2023.zip) (September 2025 release). Extract its contents into `data/nscg/` from the repository root. The ZIP already contains `pcg23Public/`; do not add another nested folder.
2. Download [Table A-2](https://ncses.nsf.gov/pubs/nsf25322/assets/technical-tables/tables/nsf25322-taba-002.xlsx) to `data/nscg/nsf25322-taba-002.xlsx`.
3. Open this notebook with that Python kernel and run all cells. O*NET inputs are already in the repository; the 25 reviewed links are embedded in Section 2.

### Sources

| Source | Version | Location |
|---|---|---|
| NSCG respondents | 2023 public-use release | `data/nscg/pcg23Public/epcg23.csv` |
| NSCG questions and labels | September 2025 documentation | `data/nscg/pcg23Public/Dpcg23.xlsx`, `Ppcg23.sas` |
| Published occupation labels | NSCG 2023 Table A-2 | `data/nscg/nsf25322-taba-002.xlsx` |
| Reviewed study concordance | 2026-10-14 | Embedded in Section 2 |
| Official SOC-to-O*NET links | SOC 2018 / O*NET-SOC 2019 | `data/onet/onetsoc2019_to_soc2018_crosswalk.xlsx` |
| Work Values, definitions and scales | O*NET 30.2 | `data/onet/db_30_2_text/` |

[NSCG release documentation](https://ncses.nsf.gov/surveys/national-survey-college-graduates/2023) · [O*NET crosswalk](https://www.onetcenter.org/taxonomy/2019/soc.html) · [O*NET Work Values 30.2](https://www.onetcenter.org/dictionary/30.2/excel/work_values.html)

## 0. Setup

In [1]:
from pathlib import Path
from io import StringIO
import hashlib
import json
import platform
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, HTML

NOTEBOOK_REL=Path('notebooks/possible data integration/nscg_onet_data_integration.ipynb')
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/NOTEBOOK_REL).is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Run from this repository or the notebook directory.')
NSCG=ROOT/'data/nscg/pcg23Public'
ONET=ROOT/'data/onet'
OUT=ROOT/'data/private/nscg_onet_work_values'
SUBSET_CSV=ROOT/'notebooks/possible data integration/nscg2023_onet30_2_subset.csv'
SOURCE={
    'survey':NSCG/'epcg23.csv',
    'dictionary':NSCG/'Dpcg23.xlsx',
    'labels':NSCG/'Ppcg23.sas',
    'hierarchy':ROOT/'data/nscg/nsf25322-taba-002.xlsx',
    'crosswalk':ONET/'onetsoc2019_to_soc2018_crosswalk.xlsx',
    'values':ONET/'db_30_2_text/Work Values.txt',
    'occupations':ONET/'db_30_2_text/Occupation Data.txt',
    'content':ONET/'db_30_2_text/Content Model Reference.txt',
    'scales':ONET/'db_30_2_text/Scales Reference.txt',
}
missing=[str(p.relative_to(ROOT)) for p in SOURCE.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing local inputs:\n- '+'\n- '.join(missing)+
                            '\nSee Data preparation above for downloads and paths.')
warnings.filterwarnings('ignore',message='Workbook contains no default style')
pd.set_option('display.max_colwidth',95)
pd.set_option('display.max_rows',35)

## 1. Source fields

`OBSNUM` uniquely identifies a row in this NSCG release. `N3OCPR` supplies the occupation key; `N3OCPRX` is retained for audit.

The 24 source fields include nine `FAC*` importance items, `JOBSATIS`, `WTSURVY`, demographics and occupation/screening fields. Job-aspect satisfaction (`SAT*`) is excluded. Complete public-file responses may include NSCG imputations.

In [2]:
dictionary=pd.read_excel(SOURCE['dictionary'])
display(dictionary.head())
definitions=dictionary.set_index('SAS_NAME')
assert definitions.index.is_unique
formats={};current=None
for line in SOURCE['labels'].read_text(encoding='cp1252').splitlines():
    start=re.match(r'\s*VALUE\s+\$?(\S+)',line)
    if start:
        current=start.group(1);formats[current]={};continue
    value=re.match(r'\s*"?([^"=]*?)"?\s*=\s*"(.*)"',line)
    if current and value:
        formats[current][value.group(1).strip()]=value.group(2).split(': ',1)[-1]
    if line.strip()==';':current=None
def labels_for(variable):
    name=definitions.at[variable,'SAS_FMT'].lstrip('$').rstrip('.')
    return formats[name]
def code_for(variable,label):
    found=[code for code,text in labels_for(variable).items() if text==label]
    assert len(found)==1,(variable,label,found)
    return found[0]
fac=dictionary.loc[dictionary.SAS_NAME.str.startswith('FAC'),'SAS_NAME'].tolist()
core=['OBSNUM','REFYR','AGE','SEX_2023','LFSTAT','JOBSATIS','N3OCPR','N3OCPRX',
      'N2OCPRMG','N3OCPRNG','OCEDRLP','DGRDG','WTSURVY','SUPWK','WAPRI']+fac
raw=pd.read_csv(SOURCE['survey'],usecols=core,dtype='string',keep_default_na=False)
display(raw.head())
survey=raw[core].apply(lambda col:col.str.strip())
assert survey.OBSNUM.is_unique and survey.OBSNUM.ne('').all()
assert survey.REFYR.eq('2023').all()
assert len(fac)==9
display(pd.DataFrame(labels_for('JOBSATIS').items(),columns=['JOBSATIS code','Label']))
display(pd.DataFrame(labels_for('FACIND').items(),columns=['FAC code','Label']))

,SAS_NAME,SESTAT_NAME,DESCRIPTION,SOURCE,QUESTION,DOMAIN,SAS_FMT
0,ACADV,N_ED_REF_WK_ENROLL_REAS_ADVANCEMT,Reason for taking courses during reference week: increase advancement opportunities,CG0: D15.6/CGN: D19.6/CGNR: D15.6,For which of the following reasons were you taking courses or enrolled? (Response to: To in...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
1,ACCAR,N_ED_REF_WK_ENROLL_REAS_BEGIN_CAREER,Reason for taking courses during reference week: further education before career,CG0: D15.1/CGN: D19.1/CGNR: D15.1,For which of the following reasons were you taking classes or enrolled? (Response to: To ga...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
2,ACCCEP,N_ED_REF_WK_EMP_ASSISTANCE,School related costs for course taking paid by employer,CG0: D16/CGN: D20/CGNR: D16,Were any of your school-related costs for taking courses paid for by an employer?,Took courses/enrolled in college/university during reference week,$YES_NO_CODES
3,ACCHG,N_ED_REF_WK_ENROLL_REAS_CHG_FLD,Reason for taking courses during reference week: change academic/occupational field,CG0: D15.3/CGN: D19.3/CGNR: D15.3,For which of the following reasons were you taking classes or enrolled? (Response to: To ch...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
4,ACDRG,N_ED_REF_WK_WORK_DEGREE,Type of degree working on during reference week,CG0: D12/CGN: D16/CGNR: D12,Toward what degree were you working?,"Took courses/enrolled in college/university during the week of February 1, 2023",$DEGREE_CODES


,AGE,DGRDG,FACADV,FACBEN,FACCHAL,FACIND,FACLOC,FACRESP,FACSAL,FACSEC,...,N3OCPR,N3OCPRNG,N3OCPRX,OBSNUM,OCEDRLP,REFYR,SEX_2023,SUPWK,WAPRI,WTSURVY
0,37,1,2,1,2,2,1,2,1,1,...,110610,11,110610,2023054942,3,2023,M,Y,06,5466.1729
1,37,2,2,1,2,1,1,2,1,1,...,121730,12,121730,2023003399,2,2023,F,N,08,46.4920
2,54,1,1,1,2,2,2,1,1,1,...,732550,73,732550,2023007536,1,2023,M,N,13,4520.0847
3,37,1,2,1,2,1,1,2,1,1,...,721510,72,721510,2023036670,2,2023,F,Y,10,9443.6334
4,37,1,3,1,2,2,1,3,1,1,...,999989,98,999989,2023031253,L,2023,F,L,L,7200.3925


,JOBSATIS code,Label
0,1,Very satisfied
1,2,Somewhat satisfied
2,3,Somewhat dissatisfied
3,4,Very dissatisfied
4,L,Logical Skip
5,,Missing


,FAC code,Label
0,1,Very important
1,2,Somewhat important
2,3,Somewhat unimportant
3,4,Not important at all
4,L,Logical Skip
5,,Missing


## 2. Reviewed occupation subset

Eligibility requires employment and a valid `JOBSATIS` category. The embedded concordance fixes the 25 study links, supervision rules and reference counts reviewed on 2026-10-14. Runtime checks compare published NSCG/SOC/O*NET titles and require one official O*NET candidate per selected SOC. Titles and occupation definitions are read from the source files.

For **bookkeeping, accounting and auditing clerks** and **sales engineers**, retain only `SUPWK = N`. First-line supervisors may require a separate SOC code; this conservative screen can exclude valid matches. The two management categories have narrower NSCG definitions. These scope decisions are study assumptions, not an official NSCG-to-SOC crosswalk or validation of individual duties.

In [3]:
# Frozen study links; labels and definitions are loaded from published files below.
concordance_csv="""NSCG code,2018 SOC Code,O*NET-SOC 2019 Code,Hold unless SUPWK N,People,Held for supervisor uncertainty,Screen retained
621440,11-9111,11-9111.00,False,743,0,743
621420,11-3021,11-3021.00,False,727,0,727
220220,19-1021,19-1021.00,False,591,0,591
520850,17-2041,17-2041.00,False,579,0,579
780310,43-3031,43-3031.00,True,455,77,378
110580,15-1244,15-1244.00,False,415,0,415
110510,15-1221,15-1221.00,False,407,0,407
121730,15-2031,15-2031.00,False,371,0,371
110570,15-1212,15-1212.00,False,361,0,361
110600,15-1254,15-1254.00,False,325,0,325
570980,41-9031,41-9031.00,True,314,91,223
110560,15-1242,15-1242.00,False,312,0,312
570900,17-2081,17-2081.00,False,249,0,249
570930,17-2131,17-2131.00,False,246,0,246
651710,15-2011,15-2011.00,False,176,0,176
442310,19-3091,19-3091.00,False,122,0,122
321920,19-2021,19-2021.00,False,119,0,119
570960,17-2161,17-2161.00,False,94,0,94
422350,19-3094,19-3094.00,False,90,0,90
570970,17-2171,17-2171.00,False,72,0,72
641020,17-3031,17-3031.00,False,66,0,66
442370,19-3041,19-3041.00,False,66,0,66
121720,15-2021,15-2021.00,False,46,0,46
570920,17-2121,17-2121.00,False,38,0,38
570950,17-2151,17-2151.00,False,33,0,33
"""
reviewed_links=pd.read_csv(StringIO(concordance_csv),dtype='string',keep_default_na=False)
display(reviewed_links.head())
scope_notes={'general': 'No extra boundary conflict identified; individual duties remain unverified.', 'management': 'NSCG asks for managers of managers. This is a subset of the O*NET management role.', 'supervisor': 'First-line supervisors can need a separate SOC code. Check SUPWK below.'}
mapping_basis='Reviewed exact-title concordance; one official SOC-to-O*NET candidate'
concordance_spec={
    'reviewed_on':'2026-10-14',
    'links':reviewed_links.to_dict(orient='records'),
    'scope_notes':scope_notes,
    'scope_rules':{'supervisor':'Hold unless SUPWK N = True',
                   'management':'Selected SOC 2018 code starts with 11-',
                   'general':'Other selected occupations'},
    'mapping_basis':mapping_basis,
}

soc_raw=pd.read_excel(SOURCE['crosswalk'],header=3,dtype=str)
display(soc_raw.head())
hierarchy_raw=pd.read_excel(SOURCE['hierarchy'],header=None)
display(hierarchy_raw.head())
occupations_raw=pd.read_csv(SOURCE['occupations'],sep='\t')
display(occupations_raw.head())

def normalize_title(value):
    return re.sub(r'[^a-z0-9]+',' ',str(value).casefold()).strip()

assert len(reviewed_links)==25 and reviewed_links['NSCG code'].is_unique
assert reviewed_links['2018 SOC Code'].is_unique and reviewed_links['O*NET-SOC 2019 Code'].is_unique
assert reviewed_links['Hold unless SUPWK N'].isin(['True','False']).all()
published=hierarchy_raw.iloc[:,[6,7]].copy()
published.columns=['Code','Title']
published=published.loc[published.Code.astype(str).str.fullmatch(r'\d{6}')]
published.Code=published.Code.astype(str)
assert published.Code.is_unique
published_titles=published.set_index('Code').Title
official=soc_raw.set_index('O*NET-SOC 2019 Code')
soc_candidates=soc_raw.groupby('2018 SOC Code')['O*NET-SOC 2019 Code'].agg(set)
occupation_definitions=occupations_raw.set_index('O*NET-SOC Code')
assert official.index.is_unique and occupation_definitions.index.is_unique
for _,row in reviewed_links.iterrows():
    code,soc,onet=row['NSCG code'],row['2018 SOC Code'],row['O*NET-SOC 2019 Code']
    expected=normalize_title(published_titles.at[code])
    assert normalize_title(labels_for('N3OCPR')[code])==expected
    assert soc_candidates.at[soc]=={onet}
    assert official.at[onet,'2018 SOC Code']==soc
    for title in [official.at[onet,'2018 SOC Title'],official.at[onet,'O*NET-SOC 2019 Title'],
                  occupation_definitions.at[onet,'Title']]:
        assert normalize_title(title)==expected
    assert isinstance(occupation_definitions.at[onet,'Description'],str)
    assert occupation_definitions.at[onet,'Description'].strip()

bridge_raw=reviewed_links.copy()
bridge_raw['NSCG title']=bridge_raw['NSCG code'].map(published_titles)
scope=np.select([bridge_raw['Hold unless SUPWK N'].eq('True'),
                 bridge_raw['2018 SOC Code'].str.startswith('11-')],
                ['supervisor','management'],default='general')
bridge_raw['Scope note']=[scope_notes[key] for key in scope]
bridge_raw['Mapping basis']=mapping_basis
bridge=bridge_raw[['NSCG code','2018 SOC Code','O*NET-SOC 2019 Code',
                   'Mapping basis','Scope note','Hold unless SUPWK N']].rename(columns={
    'NSCG code':'N3OCPR','2018 SOC Code':'SOC2018_CODE',
    'O*NET-SOC 2019 Code':'ONET_CANDIDATE_CODE','Mapping basis':'MAPPING_BASIS',
    'Scope note':'MAPPING_SCOPE_NOTE','Hold unless SUPWK N':'HOLD_UNLESS_SUPWK_N'})
bridge.HOLD_UNLESS_SUPWK_N=bridge.HOLD_UNLESS_SUPWK_N.eq('True')
assert bridge.HOLD_UNLESS_SUPWK_N.sum()==2

,NSCG code,2018 SOC Code,O*NET-SOC 2019 Code,Hold unless SUPWK N,People,Held for supervisor uncertainty,Screen retained
0,621440,11-9111,11-9111.00,False,743,0,743
1,621420,11-3021,11-3021.00,False,727,0,727
2,220220,19-1021,19-1021.00,False,591,0,591
3,520850,17-2041,17-2041.00,False,579,0,579
4,780310,43-3031,43-3031.00,True,455,77,378


,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers


,0,1,2,3,4,5,6,7
0,Table A-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Occupations used in the 2023 NSCG data tables,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(Crosswalk),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Broad occupation,NaN,Major occupation,NaN,Minor occupation,NaN,Detailed occupation,NaN
4,Code,Label,Code,Label,Code,Label,Code,Label


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and ...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to addre..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overs..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes at the local, tribal, state, or federal leve..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising policies and programs or produce collateral materia..."


In [4]:
satisfaction_labels=['Very satisfied','Somewhat satisfied','Somewhat dissatisfied','Very dissatisfied']
valid_satisfaction=[code_for('JOBSATIS',label) for label in satisfaction_labels]
valid_preferences={v:[code for code,label in labels_for(v).items()
                      if label not in ['Logical Skip','Missing']] for v in fac}
assert all(set(codes)=={'1','2','3','4'} for codes in valid_preferences.values())
employed=survey.LFSTAT.eq(code_for('LFSTAT','Employed'))
eligible_mask=employed & survey.JOBSATIS.isin(valid_satisfaction)
eligible=survey.loc[eligible_mask].copy()
assert len(eligible)==79800
assert all(eligible[v].isin(valid_preferences[v]).all() for v in fac)
assert eligible.N3OCPR.isin(labels_for('N3OCPR')).all()
eligible['NSCG_OCC_TITLE']=eligible.N3OCPR.map(labels_for('N3OCPR'))

linked=eligible.merge(bridge,on='N3OCPR',how='left',validate='many_to_one',indicator='bridge_join')
assert len(linked)==len(eligible) and linked.OBSNUM.is_unique
has_bridge=linked.bridge_join.eq('both')
supervision_hold=has_bridge & linked.HOLD_UNLESS_SUPWK_N.astype('boolean').fillna(False) &     ~linked.SUPWK.eq(code_for('SUPWK','No'))
linked['INTEGRATION_STATUS']=np.select([~has_bridge,supervision_hold],
    ['not_in_reviewed_subset','held_supervisor_uncertainty'],default='matched_reviewed')
linked['ONET_SOC_CODE']=linked.ONET_CANDIDATE_CODE.where(linked.INTEGRATION_STATUS.eq('matched_reviewed'))
assert int(has_bridge.sum())==7017 and int(supervision_hold.sum())==168

coverage=pd.Series({
    'Source respondents':len(survey),
    'Employed + valid satisfaction':len(eligible),
    'People in reviewed occupations':int(has_bridge.sum()),
    'Held for supervision uncertainty':int(supervision_hold.sum()),
    'Retained subset':int(linked.INTEGRATION_STATUS.eq('matched_reviewed').sum()),
    'Outside reviewed occupation subset':int((~has_bridge).sum()),
},name='People')
display(coverage.to_frame())

,People
Source respondents,94606
Employed + valid satisfaction,79800
People in reviewed occupations,7017
Held for supervision uncertainty,168
Retained subset,6849
Outside reviewed occupation subset,72783


**Observation:** Of 7,017 people in the reviewed categories, 168 are excluded for supervision uncertainty. The other 72,783 eligible respondents fall outside the concordance; their matchability is not assessed.

## 3. One O*NET row per occupation

The six `EX` Work Values retain their original 1–7 scale. `VH` contains rank identifiers and is excluded. Ratings are neither averaged across occupations, rescaled nor imputed.

In [5]:
values_raw=pd.read_csv(SOURCE['values'],sep='	')
display(values_raw.head())
content_raw=pd.read_csv(SOURCE['content'],sep='	')
display(content_raw.head())
scales_raw=pd.read_csv(SOURCE['scales'],sep='	')
display(scales_raw.head())

value_definitions=content_raw.loc[content_raw['Element ID'].str.fullmatch(r'1\.B\.2\.[a-f]')].copy()
value_definitions=value_definitions.sort_values('Element ID')
assert len(value_definitions)==6 and value_definitions['Element ID'].is_unique
value_definitions['Output column']='ONET_WV_'+value_definitions['Element Name'].str.lower().str.replace(r'[^a-z0-9]+','_',regex=True)
value_columns=value_definitions['Output column'].tolist()
element_columns=value_definitions.set_index('Element ID')['Output column'].to_dict()
ex_scale=scales_raw.loc[scales_raw['Scale ID'].eq('EX')].squeeze()
value_rows=values_raw.loc[values_raw['Scale ID'].eq('EX') &
    values_raw['O*NET-SOC Code'].isin(bridge.ONET_CANDIDATE_CODE)].copy()
assert len(value_rows)==25*6
assert not value_rows.duplicated(['O*NET-SOC Code','Element ID']).any()
assert value_rows['Data Value'].between(ex_scale.Minimum,ex_scale.Maximum).all()
assert value_rows.groupby('O*NET-SOC Code')['Element ID'].agg(set).map(
    lambda ids:ids==set(element_columns)).all()

value_lookup=value_rows.pivot(index='O*NET-SOC Code',columns='Element ID',values='Data Value').rename(columns=element_columns)
value_lookup=value_lookup[value_columns]
rating_month=pd.to_datetime(value_rows.Date,format='%m/%Y',errors='raise')
dated=value_rows.assign(rating_month=rating_month)
date_ranges=dated.groupby('O*NET-SOC Code').rating_month.agg(['min','max'])
value_lookup['ONET_WV_DATE_MIN']=date_ranges['min'].dt.strftime('%Y-%m')
value_lookup['ONET_WV_DATE_MAX']=date_ranges['max'].dt.strftime('%Y-%m')
value_lookup['ONET_WV_SOURCE']=value_rows.groupby('O*NET-SOC Code')['Domain Source'].agg(lambda x:'; '.join(sorted(set(x))))
value_lookup['ONET_OCC_TITLE']=value_lookup.index.map(occupation_definitions.Title)
value_lookup['ONET_WV_VERSION']='30.2'
value_lookup['ONET_WV_SCALE']='EX'
value_lookup=value_lookup.rename_axis('ONET_SOC_CODE').reset_index()
value_lookup.columns.name=None
assert value_lookup.ONET_SOC_CODE.is_unique and not value_lookup.isna().any().any()
display(value_lookup.head())
display(value_rows.groupby(['Date','Domain Source']).size().rename('Rating rows').to_frame())

,O*NET-SOC Code,Element ID,Element Name,Scale ID,Data Value,Date,Domain Source
0,11-1011.00,1.B.2.a,Achievement,EX,6.33,06/2008,Analyst
1,11-1011.00,1.B.2.b,Working Conditions,EX,6.33,06/2008,Analyst
2,11-1011.00,1.B.2.c,Recognition,EX,7.00,06/2008,Analyst
3,11-1011.00,1.B.2.d,Relationships,EX,5.00,06/2008,Analyst
4,11-1011.00,1.B.2.e,Support,EX,5.33,06/2008,Analyst


,Element ID,Element Name,Description
0,1,Worker Characteristics,Worker Characteristics
1,1.A,Abilities,Enduring attributes of the individual that influence performance
2,1.A.1,Cognitive Abilities,Abilities that influence the acquisition and application of knowledge in problem solving
3,1.A.1.a,Verbal Abilities,Abilities that influence the acquisition and application of verbal information in problem s...
4,1.A.1.a.1,Oral Comprehension,The ability to listen to and understand information and ideas presented through spoken word...


,Scale ID,Scale Name,Minimum,Maximum
0,AO,Automation,1,5
1,CF,Frequency,1,5
2,CN,Amount of Contact,1,5
3,CT,Context,1,3
4,CTP,Context (Categories 1-3),0,100


,ONET_SOC_CODE,ONET_WV_achievement,ONET_WV_working_conditions,ONET_WV_recognition,ONET_WV_relationships,ONET_WV_support,ONET_WV_independence,ONET_WV_DATE_MIN,ONET_WV_DATE_MAX,ONET_WV_SOURCE,ONET_OCC_TITLE,ONET_WV_VERSION,ONET_WV_SCALE
0,11-3021.00,5.33,6.50,5.33,4.33,5.67,5.00,2008-06,2008-06,Analyst,Computer and Information Systems Managers,30.2,EX
1,11-9111.00,5.00,6.00,4.67,6.00,5.67,6.00,2008-06,2008-06,Analyst,Medical and Health Services Managers,30.2,EX
2,15-1212.00,4.67,5.67,4.33,4.33,5.33,5.33,2008-06,2008-06,Analyst,Information Security Analysts,30.2,EX
3,15-1221.00,5.67,5.67,5.33,3.67,4.67,5.33,2008-06,2008-06,Analyst,Computer and Information Research Scientists,30.2,EX
4,15-1242.00,4.67,5.00,4.33,3.67,5.00,3.33,2008-06,2008-06,Analyst,Database Administrators,30.2,EX


,,Rating rows
Date,Domain Source,
06/2008,Analyst,132
11/2020,Analyst - Transition,18


**Observation:** Each of the 25 occupations has six ratings. Of the 150 source ratings, 132 date to June 2008 (Analyst) and 18 to November 2020 (Analyst - Transition), rather than the 2023 survey year.

## 4. Merge and validate

A left join on the screened O*NET code uses `validate='many_to_one'`. All eligible rows remain available for checks; only matched respondents enter the saved subset. Records excluded by the screen receive no ratings.

Original `JOBSATIS` and `FAC*` codes remain 1–4, with lower codes indicating greater satisfaction or importance. `WTSURVY` is unchanged.

In [6]:
integrated=linked.merge(value_lookup,on='ONET_SOC_CODE',how='left',validate='many_to_one',indicator='profile_join')
matched=integrated.INTEGRATION_STATUS.eq('matched_reviewed')
assert integrated.profile_join.eq('both').equals(matched)
assert integrated.loc[~matched,value_columns].isna().all().all()
assert not integrated.loc[matched,value_columns].isna().any().any()
assert len(integrated)==len(eligible) and integrated.OBSNUM.is_unique
derived=['NSCG_OCC_TITLE','SOC2018_CODE','ONET_SOC_CODE','ONET_OCC_TITLE',
         'MAPPING_BASIS','MAPPING_SCOPE_NOTE','INTEGRATION_STATUS',
         'ONET_WV_VERSION','ONET_WV_SCALE','ONET_WV_DATE_MIN','ONET_WV_DATE_MAX','ONET_WV_SOURCE']+value_columns
subset=integrated.loc[matched,core+derived].sort_values('OBSNUM').reset_index(drop=True)
integer_fields=['REFYR','AGE','JOBSATIS']+fac
for field in integer_fields:subset[field]=pd.to_numeric(subset[field],errors='raise').astype('Int64')
subset['WTSURVY']=pd.to_numeric(subset.WTSURVY,errors='raise').astype(float)
for field in set(subset.columns)-set(integer_fields)-set(value_columns)-{'WTSURVY'}:
    subset[field]=subset[field].astype('string')
assert len(subset)==6849 and subset.OBSNUM.is_unique
assert subset.N3OCPR.nunique()==subset.ONET_SOC_CODE.nunique()==25
assert np.isfinite(subset.WTSURVY).all() and subset.WTSURVY.gt(0).all()

# Check source preservation by release ID rather than relying on row order.
original=eligible.set_index('OBSNUM').loc[subset.OBSNUM,core[1:]].reset_index()
for field in core:
    if field in integer_fields+['WTSURVY']:
        np.testing.assert_array_equal(subset[field].to_numpy(float),pd.to_numeric(original[field]).to_numpy(float))
    else:
        assert subset[field].astype(str).tolist()==original[field].astype(str).tolist(),field
for field in value_columns:
    expected=subset.ONET_SOC_CODE.map(value_lookup.set_index('ONET_SOC_CODE')[field])
    np.testing.assert_array_equal(subset[field].to_numpy(),expected.to_numpy())

occupation_audit=bridge_raw[['NSCG code','NSCG title','O*NET-SOC 2019 Code','People',
                             'Held for supervisor uncertainty','Screen retained']].copy()
occupation_audit['Actual retained']=occupation_audit['NSCG code'].map(subset.N3OCPR.value_counts()).fillna(0).astype(int)
assert occupation_audit['Actual retained'].eq(occupation_audit['Screen retained'].astype(int)).all()
occupation_audit['Actual before screen']=occupation_audit['NSCG code'].map(
    linked.loc[has_bridge,'N3OCPR'].value_counts()).fillna(0).astype(int)
occupation_audit['Actual held']=occupation_audit['NSCG code'].map(
    linked.loc[supervision_hold,'N3OCPR'].value_counts()).fillna(0).astype(int)
assert occupation_audit['Actual before screen'].eq(occupation_audit.People.astype(int)).all()
assert occupation_audit['Actual held'].eq(occupation_audit['Held for supervisor uncertainty'].astype(int)).all()
qa=pd.Series({
    'Eligible rows before left join':len(eligible),
    'Rows after left join':len(integrated),
    'Saved subset rows':len(subset),
    'Duplicate subset IDs':int(subset.OBSNUM.duplicated().sum()),
    'Missing Work Values in subset':int(subset[value_columns].isna().sum().sum()),
    'Original fields changed':0,
    'Retained occupations':subset.ONET_SOC_CODE.nunique(),
},name='Result')
display(qa.to_frame())
display(subset[['OBSNUM','N3OCPR','ONET_SOC_CODE','JOBSATIS','FACIND','ONET_WV_independence']].head())
display(HTML('<details><summary>Retained counts for all 25 occupations</summary>'+occupation_audit.to_html(index=False)+'</details>'))

,Result
Eligible rows before left join,79800
Rows after left join,79800
Saved subset rows,6849
Duplicate subset IDs,0
Missing Work Values in subset,0
Original fields changed,0
Retained occupations,25


,OBSNUM,N3OCPR,ONET_SOC_CODE,JOBSATIS,FACIND,ONET_WV_independence
0,2023000003,621440,11-9111.00,2,1,6.00
1,2023000009,570900,17-2081.00,2,1,5.00
2,2023000065,570960,17-2161.00,3,3,5.00
3,2023000066,110580,15-1244.00,1,1,4.33
4,2023000073,621440,11-9111.00,1,1,6.00


NSCG code,NSCG title,O*NET-SOC 2019 Code,People,Held for supervisor uncertainty,Screen retained,Actual retained,Actual before screen,Actual held
621440,Medical and health services managers,11-9111.00,743,0,743,743,743,0
621420,Computer and information systems managers,11-3021.00,727,0,727,727,727,0
220220,Biochemists and biophysicists,19-1021.00,591,0,591,591,591,0
520850,Chemical engineers,17-2041.00,579,0,579,579,579,0
780310,"Bookkeeping, accounting, and auditing clerks",43-3031.00,455,77,378,378,455,77
110580,Network and computer systems administrators,15-1244.00,415,0,415,415,415,0
110510,Computer and information research scientists,15-1221.00,407,0,407,407,407,0
121730,Operations research analysts,15-2031.00,371,0,371,371,371,0
110570,Information security analysts,15-1212.00,361,0,361,361,361,0
110600,Web developers,15-1254.00,325,0,325,325,325,0


**Observation:** The join preserves all 79,800 eligible rows. The subset has unique respondent IDs, six complete O*NET ratings and unchanged source fields. Occupation counts match the concordance.

## 5. Audit and column metadata

The audit records inclusion status for all 94,606 source respondents. `not_in_reviewed_subset` identifies occupations outside the concordance, not failed matches. The column dictionary and response labels document the exported fields.

In [7]:
audit=survey[['OBSNUM','REFYR','LFSTAT','JOBSATIS','N3OCPR','N3OCPRX','SUPWK']].copy()
status_by_id=integrated.set_index('OBSNUM').INTEGRATION_STATUS
audit['INTEGRATION_STATUS']=audit.OBSNUM.map(status_by_id)
audit.loc[~employed,'INTEGRATION_STATUS']='excluded_not_employed'
audit.loc[employed & ~eligible_mask,'INTEGRATION_STATUS']='excluded_invalid_satisfaction'
audit['ONET_CANDIDATE_CODE']=audit.OBSNUM.map(integrated.set_index('OBSNUM').ONET_CANDIDATE_CODE)
audit['ONET_SOC_CODE']=audit.OBSNUM.map(integrated.set_index('OBSNUM').ONET_SOC_CODE)
assert len(audit)==len(survey) and audit.OBSNUM.is_unique
assert audit.INTEGRATION_STATUS.notna().all()
assert set(audit.loc[audit.INTEGRATION_STATUS.eq('matched_reviewed'),'OBSNUM'])==set(subset.OBSNUM)
display(audit.INTEGRATION_STATUS.value_counts().rename('People').to_frame())

derived_descriptions={
    'NSCG_OCC_TITLE':'Published label for reviewed NSCG occupation N3OCPR.',
    'SOC2018_CODE':'SOC 2018 code from the reviewed study concordance.',
    'ONET_SOC_CODE':'O*NET-SOC 2019 code assigned under the screening rule.',
    'ONET_OCC_TITLE':'Published O*NET occupation title.',
    'MAPPING_BASIS':'Basis of the study-specific concordance; not a prediction input.',
    'MAPPING_SCOPE_NOTE':'Remaining occupation-scope limitation; not a prediction input.',
    'INTEGRATION_STATUS':'Inclusion decision under this integration specification.',
    'ONET_WV_VERSION':'Version of the Work Values source table.',
    'ONET_WV_SCALE':'EX: extent rating, original 1–7 scale.',
    'ONET_WV_DATE_MIN':'Earliest source rating month among the six values (YYYY-MM).',
    'ONET_WV_DATE_MAX':'Latest source rating month among the six values (YYYY-MM).',
    'ONET_WV_SOURCE':'Source method(s) from the six rating rows.',
}
dictionary_rows=[]
for field in subset.columns:
    if field in core:
        role=('identifier' if field=='OBSNUM' else 'outcome' if field=='JOBSATIS' else
              'survey weight' if field=='WTSURVY' else 'personal preference' if field in fac else 'source covariate / audit')
        description=definitions.at[field,'DESCRIPTION'];source='NSCG 2023';element=''
        question=definitions.at[field,'QUESTION']
    elif field in value_columns:
        item=value_definitions.set_index('Output column').loc[field]
        role='occupational context';description=item.Description;source='O*NET 30.2 EX';element=item['Element ID'];question=''
    else:
        role='mapping / provenance';description=derived_descriptions[field];source='Study concordance / source metadata';element='';question=''
    dictionary_rows.append({'column':field,'dtype':str(subset[field].dtype),'role':role,
        'source':source,'element_id':element,'description':description,'question':question})
column_dictionary=pd.DataFrame(dictionary_rows)
value_labels=pd.DataFrame([
    {'column':field,'code':code,'label':label}
    for field in core if pd.notna(definitions.at[field,'SAS_FMT'])
    for code,label in labels_for(field).items()
])
assert column_dictionary.column.tolist()==subset.columns.tolist()
display(column_dictionary.loc[column_dictionary.role.isin(['outcome','personal preference','occupational context']),
                              ['column','dtype','role','element_id']])

,People
INTEGRATION_STATUS,
not_in_reviewed_subset,72783
excluded_not_employed,14806
matched_reviewed,6849
held_supervisor_uncertainty,168


,column,dtype,role,element_id
5,JOBSATIS,Int64,outcome,
15,FACADV,Int64,personal preference,
16,FACBEN,Int64,personal preference,
17,FACCHAL,Int64,personal preference,
18,FACIND,Int64,personal preference,
19,FACLOC,Int64,personal preference,
20,FACRESP,Int64,personal preference,
21,FACSAL,Int64,personal preference,
22,FACSEC,Int64,personal preference,
23,FACSOC,Int64,personal preference,


## 6. Save outputs

The integrated CSV is saved beside this notebook. A backup, respondent audit, column dictionary, response labels and manifest go to `data/private/nscg_onet_work_values/`, covered by the repository's `.gitignore`. Both CSV copies are checked for exact equality. The manifest records the embedded concordance, input hashes and output hashes.

In [8]:
import subprocess
private_files=['nscg2023_onet30_2_subset.csv','respondent_audit.csv',
               'column_dictionary.csv','value_labels.csv','manifest.json']
for name in private_files:
    result=subprocess.run(['git','check-ignore','--quiet',str(OUT/name)],cwd=ROOT)
    assert result.returncode==0,f'Expected repository ignore rule for data/private/: {name}'
OUT.mkdir(parents=True,exist_ok=True)
outputs={
    'nscg2023_onet30_2_subset.csv':subset,
    'respondent_audit.csv':audit,
    'column_dictionary.csv':column_dictionary,
    'value_labels.csv':value_labels,
}
for name,frame in outputs.items():frame.to_csv(OUT/name,index=False,float_format='%.17g')
csv_dtypes=column_dictionary.set_index('column').dtype.to_dict()
reloaded=pd.read_csv(OUT/'nscg2023_onet30_2_subset.csv',dtype=csv_dtypes,
                     keep_default_na=False,float_precision='round_trip')
pd.testing.assert_frame_equal(subset,reloaded,check_exact=True)

subset.to_csv(SUBSET_CSV,index=False,float_format='%.17g')
notebook_csv=pd.read_csv(SUBSET_CSV,dtype=csv_dtypes,
                         keep_default_na=False,float_precision='round_trip')
pd.testing.assert_frame_equal(subset,notebook_csv,check_exact=True)
assert SUBSET_CSV.read_bytes()==(OUT/SUBSET_CSV.name).read_bytes()
reloaded_audit=pd.read_csv(OUT/'respondent_audit.csv',dtype='string',keep_default_na=False)
assert len(reloaded_audit)==len(survey) and reloaded_audit.OBSNUM.is_unique
assert reloaded_audit.INTEGRATION_STATUS.value_counts().to_dict()==audit.INTEGRATION_STATUS.value_counts().to_dict()

def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
manifest={
    'pipeline_version':'nscg2023_onet30_2_v1_2026-10-14',
    'notebook':'notebooks/possible data integration/nscg_onet_data_integration.ipynb',
    'grain':'One NSCG 2023 respondent per row; OBSNUM identifies a source-file observation.',
    'mapping':'Study-specific reviewed concordance, not an official NSCG-to-SOC crosswalk.',
    'concordance':{'source':'Embedded in notebook Section 2',
        'sha256':hashlib.sha256(json.dumps(concordance_spec,sort_keys=True).encode()).hexdigest(),
        'spec':concordance_spec},
    'rules':{'employed':'LFSTAT = 1','outcome':'JOBSATIS in 1,2,3,4',
        'occupation':'25 reviewed links; use N3OCPR, not N3OCPRX',
        'supervision':'Require SUPWK = N for the two flagged categories',
        'onet':'Six distinct EX ratings per occupation, all within published scale limits',
        'merge':'Two many_to_one left joins: N3OCPR to concordance, screened ONET_SOC_CODE to Work Values; export matched subset',
        'weights':'Original WTSURVY, not normalized',
        'scores':'No fit score, scale alignment or outcome recoding'},
    'counts':{'source':len(survey),'eligible':len(eligible),'reviewed_before_screen':int(has_bridge.sum()),
        'held_supervision':int(supervision_hold.sum()),'subset':len(subset),'occupations':25},
    'sources':{key:{'path':str(path.relative_to(ROOT)),'sha256':sha256(path)} for key,path in SOURCE.items()},
    'outputs':{name:{'path':str((OUT/name).relative_to(ROOT)),'rows':len(frame),'columns':len(frame.columns),
                      'sha256':sha256(OUT/name)} for name,frame in outputs.items()},
    'notebook_subset_csv':{'path':str(SUBSET_CSV.relative_to(ROOT)),
        'rows':len(subset),'columns':len(subset.columns),'sha256':sha256(SUBSET_CSV)},
    'software':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__},
    'limitations':['Non-random occupation subset, not representative of all employed graduates.',
        'Public-file completeness can include survey imputations.',
        'Occupation-level values are shared by respondents and do not measure each actual workplace.',
        'Personal importance and occupational extent have no calibrated common scale.',
        'This integration validates keys, scope rules and data preservation, not person-level mapping accuracy.'],
}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2)+chr(10))
display(pd.DataFrame([{'File':name,'Rows':len(frame),'Columns':len(frame.columns)} for name,frame in outputs.items()]))
print(f'Integrated CSV: {SUBSET_CSV.relative_to(ROOT)}')
print('Both CSV copies reloaded with exact equality. Paths and hashes saved in manifest.json.')

,File,Rows,Columns
0,nscg2023_onet30_2_subset.csv,6849,42
1,respondent_audit.csv,94606,10
2,column_dictionary.csv,42,7
3,value_labels.csv,417,3


Integrated CSV: notebooks/possible data integration/nscg2023_onet30_2_subset.csv
Both CSV copies reloaded with exact equality. Paths and hashes saved in manifest.json.


## 7. Output and scope

The 42 columns comprise 24 NSCG fields, six Work Values and 12 mapping/provenance fields. The CSV reload matches the in-memory table exactly.

[Integrated subset CSV](nscg2023_onet30_2_subset.csv) is in this folder. The backup and supporting files are in `data/private/nscg_onet_work_values/`.

The occupation subset is non-random; its coverage does not estimate the maximum possible match rate. Individual mapping accuracy remains unvalidated. The personal and occupational scales have no calibrated common units, and this integration does not establish predictive value. Respondents share occupation ratings, so later inference must account for occupation groups.